[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_12/MFM_ch12_vol_expectations/MFM_ch12_vol_expectations.ipynb)

# MFM_ch12_vol_expectations

Case study, Lochstoer and Muir (2022), Volatility expectations and returns, Journal of Finance 77(2), 1055-1096: does the VIX react too slowly to volatility shocks? Monthly S&P 500 realised variance and month-end VIX, January 1990 - August 2026. (1) Equation 4 of the paper: VIX^2 and next month's realised variance on a + b sum_{j<12} phi^j rv_{t-j}; phi by grid NLS, moving block bootstrap standard errors (18-month blocks), on the paper's sample (to April 2020) and up to 2026. (2) Table 3: next month's realised variance, VIX^2 and the realised VRP on the real-time HAR expected variance and the weighted average of its last 6 values (weights proportional to 0.5^j), Newey-West standard errors with 12 lags. (3) Figure 2: monthly VAR(1) of expected variance, VRP = VIX^2 - expected variance and the excess return of the price index, Cholesky ordering with expected variance first; responses to a one-standard-deviation shock with 95% residual-bootstrap bands.

Expected variance comes from a HAR model on daily returns, re-estimated each month on the data available then; the paper uses 5-minute returns. The VAR omits the log price-dividend ratio of the paper, which the course data do not contain.

*Modelling Financial Markets (MFM), Chapter 12: Options and the Volatility Surface. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_12'
    os.makedirs(OUT_DIR, exist_ok=True)
    os.chdir(OUT_DIR)  # relative chart and CSV outputs persist in Drive
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats, optimize, integrate
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

N, n = stats.norm.cdf, stats.norm.pdf

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Course colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
Teal     = '#17A2B8'

HERE = '.'
SEED = 42
BASE = {'S': 100.0, 'K': 100.0, 'T': 0.25, 'r': 0.04, 'sigma': 0.2}
HEDGE = {'S0': 100.0, 'K': 100.0, 'T': 0.25, 'r': 0.04, 'sigma': 0.2, 'mu': 0.08, 'nstep': 252, 'npath': 20000}


def save_fig(name):
    """Save the figure as PDF and transparent PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend_bottom(fig, handles, ncol=3, y=0.0):
    """One legend for a multi-panel figure, below the figure."""
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def nw_mean(x, lags):
    """Mean and Newey-West (HAC) standard error of a series."""
    x = np.asarray(x, float)
    m = sm.OLS(x, np.ones_like(x)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return float(m.params[0]), float(m.bse[0])


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, pd.Timestamp):
        return str(o)
    return o

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# course data: local copy when the notebook runs inside the repository, otherwise read online
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
SNAP_DIR = next((d for d in ('.', '../Quantlets/Ch_12', '../../Quantlets/Ch_12')
                 if os.path.exists(os.path.join(d, 'ch12_deribit_snapshot.csv'))), '.')
END = '2026-09-18'

# name -> (symbol, field, type)
SERIES = {
    'sp500': ('GSPC.INDX', 'close', 'index'),
    'spy':   ('SPY.US', 'adjusted_close', 'etf'),
    'btc':   ('BTC-USD.CC', 'close', 'crypto'),
    'vix':   ('VIX.INDX', 'close', 'vol'),
    'vix9d': ('VIX9D.INDX', 'close', 'vol'),
    'vix3m': ('VIX3M.INDX', 'close', 'vol'),
    'vvix':  ('VVIX.INDX', 'close', 'vol'),
}
DERIBIT = 'https://www.deribit.com/api/v2/public/'
SNAP_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/Quantlets/Ch_12/ch12_deribit_snapshot.csv'


def read_market(symbol):
    """Read the daily series of one symbol from the course data (local copy or online)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def load_close(name, start=None, end=END):
    """Daily price (or index level); except for crypto assets, weekdays only
    (days with an unchanged close are kept: they are real trading days)."""
    symbol, col, kind = SERIES[name]
    s = read_market(symbol)[col].loc[start:end]
    s = s[s > 0].dropna()
    if kind != 'crypto':
        s = s[s.index.dayofweek < 5]
    return s.rename(name)


def log_returns(name, start=None, end=END):
    """Log returns on the series' own calendar."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


def spy_open_close(start=None, end=END):
    """Daily SPY open and close (unadjusted prices: the same-day ratio does not depend on adjustments)."""
    d = read_market('SPY.US').loc[start:end, ['open', 'close']]
    d = d[(d > 0).all(axis=1)]
    return d[d.index.dayofweek < 5]


def _get(method, **params):
    url = DERIBIT + method + '?' + '&'.join(f'{k}={v}' for k, v in params.items())
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return json.load(urllib.request.urlopen(req, timeout=60))['result']


def deribit_chain(path=None, refresh=False):
    """BTC option chain (mark prices, implied volatilities) at the snapshot time."""
    path = path or os.path.join(SNAP_DIR, 'ch12_deribit_snapshot.csv')
    if not refresh:
        for src in (path, SNAP_RAW):
            try:
                return pd.read_csv(src, parse_dates=['snapshot_utc', 'expiry'])
            except Exception:
                pass
    rows = _get('get_book_summary_by_currency', currency='BTC', kind='option')
    t0 = pd.Timestamp.now(tz='UTC').floor('min').tz_localize(None)
    out = []
    for x in rows:
        _, exp, strike, cp = x['instrument_name'].split('-')
        out.append(dict(snapshot_utc=t0, instrument=x['instrument_name'],
                        expiry=pd.to_datetime(exp, format='%d%b%y') + pd.Timedelta(hours=8),
                        strike=float(strike), type='call' if cp == 'C' else 'put',
                        mark_iv=x['mark_iv'], mark_price=x['mark_price'], bid=x['bid_price'], ask=x['ask_price'],
                        forward=x['underlying_price'], index=x['estimated_delivery_price'],
                        open_interest=x['open_interest'], volume_usd=x['volume_usd']))
    d = pd.DataFrame(out).sort_values(['expiry', 'strike', 'type']).reset_index(drop=True)
    d.to_csv(path, index=False)
    return d


def dvol_history(start='2021-03-24', end=END):
    """DVOL index (30-day implied volatility of BTC options, annualised, in %), daily values."""
    t1 = int(pd.Timestamp(end).tz_localize('UTC').timestamp() * 1000) + 86_400_000
    t0 = int(pd.Timestamp(start).tz_localize('UTC').timestamp() * 1000)
    data = []
    while True:
        r = _get('get_volatility_index_data', currency='BTC', start_timestamp=t0, end_timestamp=t1, resolution='1D')
        data += r['data']
        if not r.get('continuation') or r['continuation'] <= t0:
            break
        t1 = r['continuation']
    d = pd.DataFrame(data, columns=['t', 'open', 'high', 'low', 'close']).drop_duplicates('t')
    d.index = pd.to_datetime(d['t'], unit='ms').dt.normalize()
    return d.sort_index()['close'].loc[start:end].rename('dvol')

## Helper functions

In [ ]:
FF_DAILY = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip'


def ff_rf_daily():
    """Daily risk-free rate (Kenneth French Data Library), as a decimal."""
    import io
    import zipfile
    import urllib.request
    raw = urllib.request.urlopen(urllib.request.Request(FF_DAILY, headers={'User-Agent': 'Mozilla/5.0'}),
                                 timeout=120).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    rows = [l.split(',') for l in z.read(z.namelist()[0]).decode('latin-1').splitlines()
            if l[:8].strip().isdigit() and len(l.split(',')) == 5]
    return pd.Series([float(r[4]) / 100 for r in rows],
                     index=pd.to_datetime([r[0].strip() for r in rows], format='%Y%m%d'), name='rf')


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, pd.Timestamp):
        return str(o)
    return o


## Analysis

In [ ]:
J_LAGS = 12           # rv_t ... rv_{t-11}
BLOCK = 18            # luni (6 trimestre)
B_BOOT = 1000
PHI_WA = 0.5          # ponderile mediei expected variance (Tabelul 3)
H_IRF = 12
PAPER_END = '2020-04-30'
PAPER = dict(phi_vix=0.42, phi_vix_se=0.12,
             t3_rv=(1.53, 0.54, -0.58, 0.46, 0.469), t3_vix=(0.30, 0.12, 0.64, 0.07, 0.790),
             t3_vrp=(-1.26, 0.49, 1.23, 0.46, 0.172))
SEED = 12


def monthly():
    """Month-end VIX^2, monthly realised variance (sum of squared daily log returns, x12, in %^2 a year), the HAR
    regressors at the month end and the monthly excess return of the S&P 500 price index (in %)."""
    px = pd.concat([load_close('sp500'), load_close('vix')], axis=1, join='inner').dropna()
    r = np.log(px['sp500']).diff().dropna()
    r2 = r ** 2 * 1e4
    rf = ff_rf_daily().reindex(r.index).ffill().fillna(0.0)
    g = pd.DataFrame({'r2': r2, 'ex': np.expm1(r) - rf})
    m = pd.DataFrame({
        'rv': 12 * r2.resample('ME').sum(),
        'n': r2.resample('ME').count(),
        'vix2': px['vix'].resample('ME').last() ** 2,
        'ex': 100 * ((1 + g['ex']).resample('ME').prod() - 1),
    })
    # HAR regressors at the month end, annualised: last day, last 5 days, last 21 days
    d1 = 252 * r2.rolling(1).sum()
    d5 = 252 / 5 * r2.rolling(5).sum()
    d21 = 252 / 21 * r2.rolling(21).sum()
    for k, s in (('har_d', d1), ('har_w', d5), ('har_m', d21)):
        m[k] = s.resample('ME').last()
    m = m[m['n'] >= 15]                       # drop the incomplete last month (September 2026 ends on the 18th)
    return m.drop(columns='n')


def lag_sum(rv, phi):
    return sum(phi ** j * rv.shift(j) for j in range(J_LAGS))


def fit_phi(y, rv, grid=np.linspace(-0.6, 0.98, 317)):
    """NLS of y_t = a + b sum_j phi^j rv_{t-j}: grid over phi, OLS for (a, b)."""
    best = None
    for p in grid:
        x = lag_sum(rv, p)
        d = pd.concat([y, x], axis=1).dropna()
        X = np.column_stack([np.ones(len(d)), d.iloc[:, 1].values])
        beta, res, *_ = np.linalg.lstsq(X, d.iloc[:, 0].values, rcond=None)
        e = d.iloc[:, 0].values - X @ beta
        ssr = float(e @ e)
        if best is None or ssr < best[0]:
            yy = d.iloc[:, 0].values
            best = (ssr, float(p), beta, 1 - ssr / float(((yy - yy.mean()) ** 2).sum()), len(d))
    return dict(phi=best[1], a=float(best[2][0]), b=float(best[2][1]), r2=best[3], n=best[4])


def block_boot_phi(df, ycol, rng, B=B_BOOT):
    """Moving block bootstrap of the rows (y_t, rv_t, ..., rv_{t-11}); phi re-estimated on each sample."""
    rows = pd.concat([df[ycol]] + [df['rv'].shift(j) for j in range(J_LAGS)], axis=1).dropna().values
    n = len(rows)
    grid = np.linspace(-0.6, 0.98, 159)
    W = np.array([[p ** j for j in range(J_LAGS)] for p in grid])          # grid x lags
    out = []
    for _ in range(B):
        starts = rng.integers(0, n - BLOCK + 1, size=int(np.ceil(n / BLOCK)))
        idx = np.concatenate([np.arange(s, s + BLOCK) for s in starts])[:n]
        y, L = rows[idx, 0], rows[idx, 1:]
        X = L @ W.T                                                          # n x grid
        xc = X - X.mean(0); yc = y - y.mean()
        b = (xc * yc[:, None]).sum(0) / (xc ** 2).sum(0)
        ssr = ((yc[:, None] - xc * b) ** 2).sum(0)
        out.append(grid[int(np.argmin(ssr))])
    return float(np.std(out, ddof=1))


def har_realtime(m, burn=24):
    """Objective expected variance sigma2hat_t of next month's rv, from an expanding-window HAR regression
    estimated only on months up to t (real time); the first `burn` months are the burn-in."""
    X = sm.add_constant(m[['har_d', 'har_w', 'har_m']])
    y = m['rv'].shift(-1)
    f = pd.Series(np.nan, index=m.index)
    for i in range(burn, len(m)):
        Xi, yi = X.iloc[:i], y.iloc[:i]          # pairs (s, s+1) with s+1 <= i, i.e. known at month i
        ok = yi.notna()
        b = np.linalg.lstsq(Xi[ok].values, yi[ok].values, rcond=None)[0]
        f.iloc[i] = max(float(X.iloc[i].values @ b), 1.0)
    return f.rename('s2hat')


def wavg(s, phi=PHI_WA, k=6):
    w = np.array([phi ** (j - 1) for j in range(1, k + 1)])
    return sum(w[j - 1] * s.shift(j) for j in range(1, k + 1)) / w.sum()


def table3(m, end=None):
    d = pd.DataFrame({'s2hat': m['s2hat'], 'wa': wavg(m['s2hat']), 'rv_next': m['rv'].shift(-1),
                      'vix2': m['vix2']})
    d['vrp_real'] = d['vix2'] - d['rv_next']
    d = d.loc[:end] if end else d
    out = {}
    for y in ('vrp_real', 'rv_next', 'vix2'):
        z = d[[y, 's2hat', 'wa']].dropna()
        r = sm.OLS(z[y], sm.add_constant(z[['s2hat', 'wa']])).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
        out[y] = dict(b=float(r.params['s2hat']), b_se=float(r.bse['s2hat']), c=float(r.params['wa']),
                      c_se=float(r.bse['wa']), r2=float(r.rsquared), n=int(r.nobs),
                      start=str(z.index[0].date()), end=str(z.index[-1].date()))
    return out


def var_irf(Y, H=H_IRF):
    """VAR(1) with intercept; orthogonalised (Cholesky) responses to a 1-s.d. shock to the first variable."""
    Yv = Y.values
    X = np.column_stack([np.ones(len(Yv) - 1), Yv[:-1]])
    B = np.linalg.lstsq(X, Yv[1:], rcond=None)[0]
    E = Yv[1:] - X @ B
    S = E.T @ E / (len(E) - X.shape[1])
    P = np.linalg.cholesky(S)
    A = B[1:].T
    resp = [P[:, 0]]
    for _ in range(H):
        resp.append(A @ resp[-1])
    return np.array(resp), B, E


def irf_boot(Y, rng, B_=B_BOOT, H=H_IRF):
    base, Bc, E = var_irf(Y, H)
    Yv = Y.values
    k = Yv.shape[1]
    draws = []
    for _ in range(B_):
        e = E[rng.integers(0, len(E), size=len(E))]
        sim = np.zeros_like(Yv); sim[0] = Yv[0]
        for t in range(1, len(Yv)):
            sim[t] = Bc[0] + Bc[1:].T @ sim[t - 1] + e[t - 1]
        draws.append(var_irf(pd.DataFrame(sim, columns=Y.columns), H)[0])
    draws = np.array(draws)
    return base, np.percentile(draws, 2.5, axis=0), np.percentile(draws, 97.5, axis=0)


def fig(irf, lo, hi, cols, phis, span):
    """Left: responses to a 1-s.d. shock to expected variance (VAR on the course data);
    right: the fitted lag weights phi^j of VIX^2 and of next month's realised variance."""
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), gridspec_kw={'width_ratios': [1.45, 1]})
    ax = axes[0]
    h = np.arange(irf.shape[0])
    iv, ivrp = cols.index('s2hat'), cols.index('vrp')
    ax.axhline(0, color=Navy, lw=0.6)
    ax.fill_between(h, lo[:, ivrp], hi[:, ivrp], color=IDAred, alpha=0.15, lw=0)
    ax.plot(h, irf[:, iv], color=MainBlue, lw=1.3, marker='o', ms=2.5)
    ax.plot(h, irf[:, iv] + irf[:, ivrp], color=Teal, lw=1.1, ls='--')
    ax.plot(h, irf[:, ivrp], color=IDAred, lw=1.3, marker='o', ms=2.5)
    ax.set_xlabel('Months after the shock'); ax.set_ylabel('Response (annualised %$^2$)')
    ax.set_xticks(range(0, H_IRF + 1, 2))
    ax.set_title(f'Response to a 1-s.d. shock to expected variance, VAR(1), {span[0]}-{span[1]}', fontsize=9.5, loc='left')
    ax = axes[1]
    j = np.arange(J_LAGS)
    w = 0.38
    ax.bar(j - w / 2, phis['vix'] ** j, w * 0.95, color=Purple)
    ax.bar(j + w / 2, np.where(j == 0, 1.0, phis['rv'] ** j), w * 0.95, color=Forest)
    ax.axhline(0, color=Navy, lw=0.6)
    ax.set_xlabel('Lag $j$ of monthly realised variance'); ax.set_ylabel('Weight $\\phi^j$')
    ax.set_xticks(j[::2])
    ax.set_title(f'Weight on past realised variance, {span[2]}-{span[1]}', fontsize=9.5, loc='left')
    handles = [Line2D([], [], color=MainBlue, lw=1.3, marker='o', ms=2.5, label='Expected variance (real-time HAR)'),
               Line2D([], [], color=Teal, lw=1.1, ls='--', label='VIX$^2$ = expected variance + VRP'),
               Line2D([], [], color=IDAred, lw=1.3, marker='o', ms=2.5, label='VRP = VIX$^2$ $-$ expected variance'),
               Patch(color=IDAred, alpha=0.15, label='VRP: 95% bootstrap band'),
               Patch(color=Purple, label=f'VIX$^2$: $\\hat\\phi$ = {phis["vix"]:.2f}'),
               Patch(color=Forest, label=f'Next month realised variance: $\\hat\\phi$ = {phis["rv"]:.2f}')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=3, frameon=False, fontsize=7.6)
    plt.tight_layout()
    save_fig('ch12_lm_vrp_irf')


def main():
    rng = np.random.default_rng(SEED)
    m = monthly()
    m['s2hat'] = har_realtime(m)
    m['vrp'] = m['vix2'] - m['s2hat']
    res = dict(start=str(m.index[0].date()), end=str(m.index[-1].date()), n=int(len(m)), paper=PAPER)
    # 1. phi (eq. 4), paper sample and full sample
    for tag, end in (('paper', PAPER_END), ('full', None)):
        mm = m.loc[:end]
        fv = fit_phi(mm['vix2'], mm['rv'])
        fr = fit_phi(mm['rv'].shift(-1), mm['rv'])
        fv['se'] = block_boot_phi(mm.assign(y=mm['vix2']), 'y', rng)
        fr['se'] = block_boot_phi(mm.assign(y=mm['rv'].shift(-1)), 'y', rng)
        res['phi_' + tag] = dict(vix=fv, rv=fr)
    # 2. Table 3 analogue
    res['t3_paper'] = table3(m, PAPER_END)
    res['t3_full'] = table3(m)
    # 3. VAR impulse responses, full sample from the first month with sigma2hat
    Y = m[['s2hat', 'vrp', 'ex']].dropna()
    irf, lo, hi = irf_boot(Y, rng)
    cols = list(Y.columns)
    iv, ivrp, ie = (cols.index(c) for c in ('s2hat', 'vrp', 'ex'))
    neg = [int(k) for k in range(H_IRF + 1) if hi[k, ivrp] < 0]
    res['irf'] = dict(start=str(Y.index[0].date()), end=str(Y.index[-1].date()), n=int(len(Y)),
                      s2hat=irf[:, iv].tolist(), vrp=irf[:, ivrp].tolist(), vrp_lo=lo[:, ivrp].tolist(),
                      vrp_hi=hi[:, ivrp].tolist(), ex=irf[:, ie].tolist(), ex_lo=lo[:, ie].tolist(),
                      ex_hi=hi[:, ie].tolist(), vrp_sig_neg=neg,
                      vrp_first_pos=next((int(k) for k in range(H_IRF + 1) if irf[k, ivrp] > 0), None),
                      vrp_min=float(irf[:, ivrp].min()), vrp_min_h=int(np.argmin(irf[:, ivrp])),
                      shock=float(irf[0, iv]))
    irf_p, lo_p, hi_p = irf_boot(m.loc[:PAPER_END, ['s2hat', 'vrp', 'ex']].dropna(), rng)
    res['irf_paper'] = dict(vrp=irf_p[:, 1].tolist(), vrp_lo=lo_p[:, 1].tolist(), vrp_hi=hi_p[:, 1].tolist(),
                            s2hat=irf_p[:, 0].tolist())
    # descriptive: mean VRP, share of negative months
    v = m['vrp'].dropna()
    res['vrp_desc'] = dict(mean=float(v.mean()), share_neg=float((v < 0).mean()), n=int(len(v)),
                           corr_s2_vix2=float(m[['s2hat', 'vix2']].dropna().corr().iloc[0, 1]))
    fig(irf, lo, hi, cols, dict(vix=res['phi_full']['vix']['phi'], rv=res['phi_full']['rv']['phi']),
        (res['irf']['start'][:4], res['end'][:4], res['start'][:4]))
    # series for the interactive page
    res['series'] = dict(dates=[str(d.date()) for d in m.index], rv=m['rv'].round(3).tolist(),
                         vix2=m['vix2'].round(3).tolist(),
                         s2hat=[None if np.isnan(x) else round(float(x), 3) for x in m['s2hat']],
                         ex=m['ex'].round(4).tolist())
    with open(os.path.join(HERE, 'ch12_lm_case.json'), 'w') as f:
        json.dump(jsonable(res), f, indent=1)
    print(json.dumps({k: v for k, v in jsonable(res).items() if k != 'series'}, indent=1)[:6000])


## Run

In [ ]:
main()


![ch12_lm_vrp_irf](./ch12_lm_vrp_irf.png)

Output: `ch12_lm_vrp_irf.pdf`